# Bug Severity Prediction — Implementation Walkthrough

This notebook walks through the complete pipeline for predicting bug severity using CodeBERT and KICL.

## Contents

1. Dataset Exploration
2. Class Imbalance Analysis
3. Model Architecture
4. Training with Imbalance Fixes
5. Evaluation & Results
6. KICL Pipeline Overview

## 1. Dataset Exploration

In [ ]:
import json
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

# Load training data
train_data = [json.loads(line) for line in open('../data/train.jsonl')]
valid_data = [json.loads(line) for line in open('../data/valid.jsonl')]
test_data = [json.loads(line) for line in open('../data/test.jsonl')]

print(f'Train: {len(train_data)} samples')
print(f'Valid: {len(valid_data)} samples')
print(f'Test:  {len(test_data)} samples')
print(f'Total: {len(train_data) + len(valid_data) + len(test_data)} samples')

In [ ]:
# Sample data point
sample = train_data[0]
print('Fields:', list(sample.keys()))
print(f'\nProject: {sample["project_name"]} v{sample["project_version"]}')
print(f'Label: {sample["label"]}')
print(f'\nCode (first 300 chars):\n{sample["code"][:300]}...')

## 2. Class Imbalance Analysis

This is the **core challenge** of this project. The dataset is heavily skewed toward the "Major" severity class.

In [ ]:
severity_names = {0: 'Critical', 1: 'Major', 2: 'Medium', 3: 'Minor'}
train_labels = [d['label'] for d in train_data]
label_counts = Counter(train_labels)

print('Training Set Class Distribution:')
print('-' * 40)
for label in sorted(label_counts.keys()):
    count = label_counts[label]
    pct = count / len(train_labels) * 100
    bar = '█' * int(pct)
    print(f'  {severity_names[label]:>8} ({label}): {count:>5} ({pct:5.1f}%) {bar}')

# Visualization
fig, ax = plt.subplots(figsize=(8, 5))
labels = [severity_names[i] for i in range(4)]
counts = [label_counts[i] for i in range(4)]
colors = ['#e74c3c', '#e67e22', '#f39c12', '#27ae60']
bars = ax.bar(labels, counts, color=colors, edgecolor='white', linewidth=2)
ax.set_ylabel('Number of Samples', fontsize=12)
ax.set_title('Severity Class Distribution (Training Set)', fontsize=14, fontweight='bold')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{count}', ha='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nImbalance ratio (Major/Critical): {label_counts[1]/label_counts[0]:.1f}x')

## 3. Model Architecture

### Baseline: CodeBERT → [CLS] → Classification Head

```
CodeBERT Encoder (124M params)
        ↓
[CLS] Token (768-dim)
        ↓
Dropout(0.1) → Dense(768→768) → Tanh → Dropout(0.1)
        ↓
Linear(768→4) → Softmax
```

### Key Design Decisions

1. **Weighted CrossEntropyLoss** — inverse-frequency class weights
2. **WeightedRandomSampler** — ensures balanced batches
3. **Focal Loss option** — down-weights easy examples (γ=2.0)

In [ ]:
import sys
sys.path.insert(0, '../scripts')
from model import CodeBERTClassifier

# Show model architecture
model = CodeBERTClassifier(num_labels=4, loss_type='weighted_ce')
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:>12,}')
print(f'Trainable parameters: {trainable_params:>12,}')
print(f'\nClassification Head:')
print(f'  Dense:  768 → 768')
print(f'  Output: 768 → 4')

## 4. Training with Imbalance Fixes

### The Collapse Problem

Without imbalance handling, the model converges to predicting **only** the majority class (Major).

### Our Fixes

| Fix | Description |
|-----|-------------|
| Weighted CE Loss | `weight_i = n_samples / (n_classes × count_i)` |
| WeightedRandomSampler | Oversamples minority classes per batch |
| Learning Rate = 1e-5 | Lower than default 2e-5 |
| Gradient Accumulation | Effective batch size = 32 |
| Prediction Monitoring | Track class diversity every epoch |

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

train_labels_np = np.array(train_labels)
weights = compute_class_weight('balanced', classes=np.arange(4), y=train_labels_np)

print('Computed Class Weights (sklearn balanced):')
print('-' * 45)
for i, (name, w) in enumerate(zip(labels, weights)):
    count = label_counts[i]
    print(f'  {name:>8}: weight = {w:.4f}  (count = {count})')
print(f'\nRatio (max/min weight): {max(weights)/min(weights):.1f}x')

## 5. Evaluation & Results

In [ ]:
# Load training history
with open('../models/training_history.json') as f:
    history = json.load(f)

print(f'Trained for {len(history)} epochs')
print(f'Best validation loss: epoch {min(history, key=lambda h: h["val_loss"])["epoch"]}')
print()

# Show epoch-by-epoch progress
print(f'{"Epoch":>5} {"Train Loss":>11} {"Val Loss":>9} {"Val Acc":>8} {"Classes":>8}')
print('-' * 50)
for h in history:
    print(f'{h["epoch"]:>5} {h["train_loss"]:>11.4f} {h["val_loss"]:>9.4f} '
          f'{h["val_acc"]:>8.4f} {h["classes_predicted"]:>8}')

In [ ]:
# Load final evaluation results
with open('../results/baseline_results.json') as f:
    results = json.load(f)

print('Final Test Set Results')
print('=' * 40)
print(f'  Precision (Weighted): {results["precision_weighted"]:.4f}')
print(f'  Recall (Weighted):    {results["recall_weighted"]:.4f}')
print(f'  F1 (Weighted):        {results["f1_weighted"]:.4f}')
print(f'  AUC (Weighted):       {results["auc_weighted"]:.4f}')
print(f'  MCC:                  {results["mcc"]:.4f}')
print()
print('Per-Class F1:')
for cls, f1 in results['f1_per_class'].items():
    label_idx = int(cls.split('_')[1])
    print(f'  {severity_names[label_idx]:>8}: {f1:.4f}')

In [ ]:
# Display confusion matrix
from IPython.display import Image, display
display(Image(filename='../results/confusion_matrix.png', width=500))

In [ ]:
# Display training curves
display(Image(filename='../results/training_curves.png', width=800))

## 6. KICL Pipeline Overview

The KICL (Knowledge-Intensified Contrastive Learning) approach extends CodeBERT with a three-stage pre-training pipeline:

### Stage 1: Knowledge-Intensified MLM (KI-MLM)
- Masks **50%** of tokens (vs. standard 15%)
- Forces the model to learn project-specific patterns and code semantics

### Stage 2: Supervised Contrastive Learning (SupCon)
- Positive pairs: samples with the **same** severity label
- Negative pairs: samples with **different** severity labels
- Learns to cluster similar-severity code in embedding space

### Stage 3: Classification Fine-tuning
- Uses weighted CrossEntropyLoss with the contrastively pre-trained encoder
- Classification head: Dense → Tanh → Linear → 4-class Softmax

In [ ]:
from kicl_model import KICLModel

kicl = KICLModel(num_labels=4)
total = sum(p.numel() for p in kicl.parameters())
print(f'KICL Model Parameters: {total:,}')
print(f'\nHeads:')
print(f'  MLM Head:        {sum(p.numel() for p in kicl.mlm_head.parameters()):,} params')
print(f'  Projection Head: {sum(p.numel() for p in kicl.projection_head.parameters()):,} params')
print(f'  Classifier:      {sum(p.numel() for p in kicl.dense.parameters()) + sum(p.numel() for p in kicl.classifier.parameters()):,} params')

---

## Summary

| Aspect | Details |
|--------|---------|
| **Task** | 4-class bug severity prediction from source code |
| **Model** | CodeBERT (124M params) + classification head |
| **Key Challenge** | Severe class imbalance (63% Major class) |
| **Solution** | Weighted CE + WeightedRandomSampler + low LR |
| **MCC** | 0.3049 (from ~0 baseline) |
| **AUC** | 0.7300 |
| **Best F1** | 0.9143 (Medium class) |